# **1. Problem Formulation - Adversarial Contextual Bandit**

![protocol](materials/adversarial_contextual_bandit_interaction_protocol.png)

> **📌(1) Remark: Notice that $(x_t)_{t=1}^n$ and $(c_t)_{t=1}^n$ are chosen before the game start !**
>
> which means that the adversary and contexts are **offline** in this env!

## **1.1 Environment**

- Oblivious Adversary
- Oblivious Contexts

| Symbol | Definition |
|-------|------------|
| $k \in \mathbb{N}_+$  ($k < \infty$)| total number of arms  |
| $n \in \mathbb{N}_+$  | time horizon   |
| $C$ |  a set of possible contexts |

## **1.2 Decision Variables**

| Symbol | Definition |
|-------|------------|
| $A_t \in [k]$ | The arm chosed at step $t$|

## **1.3 🚨 Adversarial Reward**

suppose the adversary has the full information about the policy, then secretly chooses the reward vector $x_t \in [0,1]^k$ for step $t$, inorder to minimize the accumulated reward 

$$
\sum_{t=1}^n x_{t,A_t} \in \mathbb{R}
$$

then we have a sequence of adversarial reward vectors $(x_t)_{t=1}^n$ along $n$ steps. 

## **1.4 🚨 Adversarial Context**

The adversary also secrectly chooses a sequence of context $(c_t)_{t=1}^n$, where $c_t \in C$

> **❓Question: How $c_t$ is chosen?**
> 
> It is clear that how the adversary chooses $x_t$ to minimize the accumulated reward $\sum_{t=1}^n x_{t,A_t}$, but the choice of $c_t$ seems not directly influencing $\sum_{t=1}^n x_{t,A_t}$, then what is the specific strategy for the adversary to choose $c_t$ ?

## **1.5 Regret**

$$
R_n = \mathbb{E}\left[\sum_{c \in C} \max \limits_{i \in [k]} \sum_{t \in [n]: c_t = c}(x_{ti} - X_t)\right]
$$


In a nutshell, this regret is defined by comparing the total reward of the policy achieved $\sum_{t=1}^n X_t$,with the best reward that an expert defined as $\phi: C \rightarrow [k]$ to achieve

> **📌 (1) rationale behind this formula:**
>
> - $\arg\max \limits_{i \in [k]} \sum_{t \in [n]: c_t = c}(x_{ti} - X_t)$ is **the arm** that maximize total regret under **the context** $c$, then add up the acumulated regrets of each $c \in C$, taking expectation,  get $R_n$. What this part actually does is that it finds the best arm under context $c$.
> - notice that $R_n = \mathbb{E}\left[\sum_{c \in C} \max \limits_{i \in [k]} \sum_{t \in [n]: c_t = c}(x_{ti} - X_t)\right] \le \mathbb{E}\left[\sum_{c \in C}  \sum_{t \in [n]: c_t = c}((\max \limits_{i \in [k]}x_{ti}) - X_t)\right]$, **the intention of using the benchmark $R_n$ rather than $\mathbb{E}\left[\sum_{c \in C}  \sum_{t \in [n]: c_t = c}((\max \limits_{i \in [k]}x_{ti}) - X_t)\right]$ is shown in the next remark.**
> - The "best arm" defined in this benchmark is actually different from that is defined in the stochastic bandit problems, the latter one define the best arm as $i^* := \max\limits_{i \in [k] }\mu_i$

> **📌 (2) The reason to design the benchmark like this**
>
> This $R_n$ restricts that one can only choose the same arm under the same context $c$, this align with the machanism of the expert that we realize it by a map $\phi: C \rightarrow [k]$, by which, any expert can only choose the same arm under the same context, thus, **🟥$\arg\max \limits_{i \in [k]} \sum_{t \in [n]: c_t = c}x_{ti}$ is indeed the greatest reward that an expert defined as $\phi: C \rightarrow [k]$ can achieve**, thus the regret can not be defined as  $\mathbb{E}\left[\sum_{c \in C}  \sum_{t \in [n]: c_t = c}((\max \limits_{i \in [k]}x_{ti}) - X_t)\right]$, beacause this requires the expert to choose different arms under the same context, which is impossible for any well-defined map from $C$ to $[k]$.



# **2. Bandits with Expert Advice**

## **2.1 Contexts Cluster**

### **Partition**

as was discussed in 3.1.(2), using one bandit algorithm per context may lead to a poor choice because the additional precision is wasted unless the amount of data is enormous, thus we may use a partition trick to package contexts into several groups, then assign one bandit algorithm per group:

Let $\mathcal{P} \subseteq 2^C$ be a partition of $C$, i.e. $\cap_{i \in S }P_i = \empty$  for $\forall S\subseteq \mathcal{P}$ and $\cup_{i \in P} P_i = \mathcal{P}$, then group thoses context into $\mathcal{P}$.

### **Similarity Function**

- **Design a measure of similarity:** Let s:$C \times C \rightarrow [0,1]$ be a function measuring the similarity between pairs of contexts on [0,1]-scale 

> 🚨 Notice that $s$ is not the distance, it's similarity, i.e. similar contexts pair gets **high value** of $s$

> 📌 Maybe $s$ should be more specificly defined s.t. $s(c,d) = s(d,c)$ for $\forall c,d \in C$

- **Design a Regularization for Context Separation:** difine the penalty of "similar contexts but different $\phi$" to be:
$$g(\phi) = \frac{1}{|\mathcal{C}|^2}\sum_{(c,d) \in \mathcal{C}\times \mathcal{C}}s(c,d) \,\mathbb{I}\{\phi(c) \neq \phi(d)\}$$

then let 

$$\Phi_{\theta}:= \{\phi: C \rightarrow [k] \,|\, g(\phi)  \le \theta \}$$

where $\theta \in (0,1)$ is a user-tuned threshhold

> 📌 Remark: $\Phi_{\theta}$ garantees that maps from it is more possible to assign similar context the same value. 

##  **2.2 Bandit with Expert Advice Framework**


![image](materials/bandit_with_expert_protocol.png)

- 1. Adversary secretely chooses rewards $x \in [0,1]^{n \times k}$ **before the game start**
- 2. Experts secretly choose predictions $E^{(1)}, \cdots, E^{(n)}$ **before the game start**
- 3. **Game start:** For rounds $t = 1,2, \cdots, n:$

 
 $\cdots$ $\overset{(\mathcal{H}_{t-1},c_t) \in \Omega \times C}{\longrightarrow}$ Experts $\overset{E^{(t) \in [0,1]^{M \times k}}}{\longrightarrow}$ Learner $\overset{P_t \in \mathcal{P}_{k-1}}{\longrightarrow}$ Random Selector $\overset{A_t \sim P_t}{\longrightarrow}$ Reward Map $\overset{X_t = x_{t,A_t}}{\longrightarrow}$ $\cdots$

 where $\mathcal{H}_t:=(a_1, x_1, \cdots, a_t, x_t)$ is the decision and reward history

 > **📌Remark: The experts are actually doing dimension reduction**
>
> we may deem the Experts as a map $e: \Omega \times C \rightarrow [0,1]^{M \times k}$ ,$ (\mathcal{H}_{t-1},c_t) \mapsto E^{(t)}$, i.e. the experts group takes decision history along with the context $c_t$ and processes the information and makes different decision advice to the learner.

# **3. Algorithm-Exp4**

![image](materials/algo_exp4.png)

> **📌Remark: The experts are actually doing dimension reduction**
>
> The only decision variable here is the $Q_t \in [0,1]^M$, which decide the distro of choosing experts 

## **3.1 Algorithm Analysis** 

### **3.1.1 $P_t = Q_t E^{(t)}$**

$proof.$

$$
\begin{align}
\mathbb{P}(\text{Exp4 chooses arm } i)
&= \mathbb{P}(\cup_{m \in [M]}A_{m,i}) \\
&= \sum_{m=1}^M 
\mathbb{P}(\text{choose arm } i \mid \text{choose expert } m)
\cdot
\mathbb{P}(\text{choose expert } m) \\
&= \sum_{m=1}^M 
E^{(t)}_{m,i} \, (Q_t)_m \\
&= 
\begin{bmatrix}
(Q_t)_1 & \cdots & (Q_t)_M
\end{bmatrix}
\begin{bmatrix}
E^{(t)}_{1,i} \\
\vdots \\
E^{(t)}_{M,i}
\end{bmatrix}
\end{align}
$$

where $A_{m,i}:=\{ \text{choose arm i advised by expert m}\}$, then it's obvious that 
$$
\begin{align}
P_t &= (\mathbb{P}(\text{Exp4 chooses arm } 1), \cdots, \mathbb{P}(\text{Exp4 chooses arm } k)) \\
&= Q_t E^{(t)}
\end{align}
$$

complete the proof. $\blacksquare$

### **3.1.2 The rationale of using $\hat{X}_{t i}=1 - \frac{\mathbb{I}\{A_t = i\}}{P_{t i} + \gamma}\,(1 - X_t)$**

One question is that why using  $\hat{X}_{t i}=1 - \frac{\mathbb{I}\{A_t = i\}}{P_{t i} + \gamma}\,(1 - X_t)$ instead of  $\hat{X}_{t i}=\frac{\mathbb{I}\{A_t = i\}}{P_{t i} + \gamma}\,X_t$ as the estimator. Some reason may be as follow:

for the convinience of reference, let $\hat{Y}_{ti}:= \frac{\mathbb{I}\{A_t = i\}}{P_{t i} + \gamma}\,X_t$
- **Bounds**: if $\gamma >0$, then $-\infty < 1- \frac{1}{\gamma} \le \hat{X}_{ti} \le 1$, while $\hat{Y}_{ti} \in [0,\frac{1}{\gamma}]$,thus if $\eta \in (0, \infty)$,then $\exp(\eta \hat{X}_{ti}) \in (0,e]$ for $\forall \gamma >0$, which is a relatively small interval, while $\exp(\eta \hat{X}_{ti}) \in [1,e^{1/\gamma}]$, where $e^{1/\gamma}$ could be very large if $\gamma \rightarrow 0^+$

- **Bias**: since $\mathbb{E}[\hat{X}_{ti}] = \left(1 - \frac{1-x_{ti}}{P_{ti} + \gamma} \right)P_{ti} + (1- P_{ti}) = 1 - \frac{1-x_{ti}}{P_{ti} + \gamma}P_{ti} = \frac{\gamma + P_{ti}x_{ti}}{P_{ti}+ \gamma}$, then if $\gamma$ is sufficiently close to 0,  $\hat{X}_{ti}$ is **nearly unbiased**. And it has been previously proved that $\hat{Y}_{ti}$ is unbiased.


### **3.1.3 The rationale of setting $\tilde{X}_t = E^{(t)}\hat{X}_t$**

$$
\begin{align}
E^{(t)} \hat{X}_t 
&= 
\begin{bmatrix} 
E^{(t)}_{1,1} & E^{(t)}_{1,2} & \cdots & E^{(t)}_{1,k} \\
E^{(t)}_{2,1} & E^{(t)}_{2,2} & \cdots & E^{(t)}_{2,k} \\
\vdots & \vdots & \cdots & \vdots \\
E^{(t)}_{M,1} & E^{(t)}_{M,2} & \cdots & E^{(t)}_{M,k} \\
\end{bmatrix} 
\begin{bmatrix}
\hat{X}_{t,1} \\
\hat{X}_{t,2} \\
\vdots \\
\hat{X}_{t,k}
\end{bmatrix} \\
&= 
\begin{bmatrix}
\sum_{i=1}^k E^{(t)}_{1,i}\hat{X}_{ti} \\
\sum_{i=1}^k E^{(t)}_{2,i}\hat{X}_{ti} \\
\vdots \\
\sum_{i=1}^k E^{(t)}_{M,i}\hat{X}_{ti} \\
\end{bmatrix} \\
&= 
\begin{bmatrix}
E^{(t)}_{1,A_t}\hat{X}_{t,A_t} \\
E^{(t)}_{2,A_t}\hat{X}_{t,A_t} \\
\vdots \\
E^{(t)}_{M,A_t}\hat{X}_{t,A_t} \\
\end{bmatrix} \\
\end{align}
$$

so $\tilde{X}_t$ transfers the reward estimation of arms $\begin{bmatrix}
\hat{X}_{t,1} &
\hat{X}_{t,2} &
\cdots &
\hat{X}_{t,k}
\end{bmatrix}^T $ into the reward estimation of experts, and obviously, experts that assigns arm $A_t$ with high probability (i.e. $E^{(t)}_{m,A_t} \in [0,1]$ is large),say, expert $m$, will get relatively significant renew since $|(\tilde{X}_t)_m|$ is large. 

### **3.1.4 The rationale of the exponential weighting** 

notice that $Q_{t+1,i} =\frac{\exp\!\left(\eta \tilde{X}_{t i}\right) Q_{t i}}{\sum_{j=1}^{M}\exp\!\left(\eta \tilde{X}_{t j}\right) Q_{t j}}$, which is a little bit different from the exponential weighting in Exp3, the reason is left for further discussion.

![algo-exp3](materials/algo-exp3.png)

### **3.1.5 Regret Analysis**

Let 

- $\gamma = 0$
- $\eta  = \sqrt{\frac{2\log(M)}{nk}}$

Then 

$$
R_n := \mathbb{E}[\max\limits_{m \in [M]} \sum_{t=1}^n E^{(t)}_m\cdot x_{t} - \sum_{t=1}^n X_t] \le \sqrt{2nk \log(M)}
$$

$proof.$

Let 
- $\mathcal{F}_t := \sigma (E^{(1)}, A_1, E^{(2)}, A_2, \cdots, A_{t-1}, E^{(t)})$
- $\mathbb{E}_t[\cdot] := \mathbb{E}_t[\cdot | \mathcal{F}_t]$
- $m^* := \arg\max_{m \in [M]} \sum_{t=1}^n E^{(t)}_m x_t$

Then we have:

$$
\sum_{t=1}^n (\tilde{X}_t)_{m^*} - \sum_{t=1}^n Q_t \cdot \tilde{X}_t \le \frac{\log(M)}{\eta} + \frac{\eta}{2} \sum_{t=1}^n  Q_t\cdot (\vec{1}-\tilde{X}_{t})^{(2)}
$$

where $Q_t \in [0,1]^M$ is a row vector, $\tilde{X}_t \in \mathbb{R}^{(M)}$ is a column vector, $\vec{1} \in \mathbb{R}^{(M)}$, and for $X \in \mathbb{R}^n$, $(X)^{(2)}:= (X_1^2, X_2^2, \cdots, X_n^2)$

Notice that we set $\gamma = 0$ here, so as discussed in 3.1.2, $\hat{X}_{tm}$ is unbiased in this case, hence we have $\mathbb{E}_t[\tilde{X}_{t}] = \mathbb{E}_t[E^{(t)}\hat{X}_t] = E^{(t)}x_t$.

Then, by taking expectation to the inequality, we have:

$$
\sum_{t=1}^n E^{(t)}_{m^*}x_t - \sum_{t=1}^n Q_t \cdot E^{(t)}x_t \le \frac{\log(M)}{\eta} + \frac{\eta}{2} \sum_{t=1}^n \mathbb{E}_t [Q_t\cdot (\vec{1}-\tilde{X}_{t})^{(2)}]
$$

and by 3.1.1, the left hand side of this inequation is equal to $\sum_{t=1}^n E^{(t)}_{m^*}x_t - \sum_{t=1}^n P_t x_t = R_n$

thus we eventualy have 

$$
R_n \le \frac{\log(M)}{\eta} + \frac{\eta}{2} \sum_{t=1}^n \mathbb{E}_t [Q_t\cdot (\vec{1}-\tilde{X}_{t})^{(2)}]
$$



Now, to handle the expectation on the right hand side, let $z_{ti} = 1- x_{ti}$,$Z_t = 1- X_t$ ,$\hat{Z}_t = \vec{1} - \hat{X}_t$ ,then $\tilde{Z} := E^{(t)}\hat{Z}_t = E^{(t)}\vec{1} - E^{(t)}\hat{X}_t = \vec{1} - \tilde{X}_t$

Then

$$
\begin{align}
\hat{Z}_{t} &= \vec{1} - 
\begin{bmatrix}
1 - \frac{\mathbb{I}\{A_t = 1\}}{P_{t,1}}(1-X_t) \\
1 - \frac{\mathbb{I}\{A_t = 2\}}{P_{t,2}}(1-X_t) \\
\vdots \\
1 - \frac{\mathbb{I}\{A_t = k\}}{P_{t,k}}(1-X_t) \\
\end{bmatrix} \\
&= 
\begin{bmatrix}
\frac{\mathbb{I}\{A_t = 1\}}{P_{t,1}}Z_t \\
\frac{\mathbb{I}\{A_t = 2\}}{P_{t,2}}Z_t \\
\vdots \\
\frac{\mathbb{I}\{A_t = k\}}{P_{t,k}}Z_t \\
\end{bmatrix} \\
\end{align}
$$



and consequently

$$
\begin{align}
E^{(t)} \hat{Z}_t 
&= 
\begin{bmatrix} 
E^{(t)}_{1,1} & E^{(t)}_{1,2} & \cdots & E^{(t)}_{1,k} \\
E^{(t)}_{2,1} & E^{(t)}_{2,2} & \cdots & E^{(t)}_{2,k} \\
\vdots & \vdots & \cdots & \vdots \\
E^{(t)}_{M,1} & E^{(t)}_{M,2} & \cdots & E^{(t)}_{M,k} \\
\end{bmatrix} 
\begin{bmatrix}
\frac{\mathbb{I}\{A_t = 1\}}{P_{t,1}}Z_t \\
\frac{\mathbb{I}\{A_t = 2\}}{P_{t,2}}Z_t \\
\vdots \\
\frac{\mathbb{I}\{A_t = k\}}{P_{t,k}}Z_t \\
\end{bmatrix}\\
&= 
\begin{bmatrix}
\sum_{i=1}^k E^{(t)}_{1,i}\frac{\mathbb{I}\{A_t = i\}}{P_{t,i}}Z_t\\
\sum_{i=1}^k E^{(t)}_{2,i}\frac{\mathbb{I}\{A_t = i\}}{P_{t,i}}Z_t \\
\vdots \\
\sum_{i=1}^k E^{(t)}_{M,i}\frac{\mathbb{I}\{A_t = i\}}{P_{t,i}}Z_t \\
\end{bmatrix} \\
&= 
\begin{bmatrix}
E^{(t)}_{1,A_t}\frac{(z_t)_{{\,}_{A_t}}}{P_{t,A_t}} \\
E^{(t)}_{2,A_t}\frac{(z_t)_{{\,}_{A_t}}}{P_{t,A_t}} \\
\vdots \\
E^{(t)}_{M,A_t}\frac{(z_t)_{{\,}_{A_t}}}{P_{t,A_t}} \\
\end{bmatrix}
\end{align}
$$

Thus 
$$
\begin{align}
\mathbb{E}_t [\tilde{Z}_{t}^{(2)}]
&= \begin{bmatrix}
\mathbb{E}_t [\left(E^{(t)}_{1,A_t}\frac{(z_t)_{{\,}_{A_t}}}{P_{t,A_t}}\right)^2 ]\\
\mathbb{E}_t [\left(E^{(t)}_{2,A_t}\frac{(z_t)_{{\,}_{A_t}}}{P_{t,A_t}}\right)^2] \\
\vdots \\
\mathbb{E}_t [\left(E^{(t)}_{M,A_t}\frac{(z_t)_{{\,}_{A_t}}}{P_{t,A_t}}\right)^2] \\
\end{bmatrix}\\
&= \begin{bmatrix}
\sum_{i=1}^k \frac{\left(E^{(t)}_{1,i}z_{t,i}  \right)^2}{P_{t,i}}\\
\sum_{i=1}^k \frac{\left(E^{(t)}_{2,i}z_{t,i}  \right)^2}{P_{t,i}} \\
\vdots \\
\sum_{i=1}^k \frac{\left(E^{(t)}_{m,i}z_{t,i}  \right)^2}{P_{t,i}} \\
\end{bmatrix}\\
&\le \begin{bmatrix}
\sum_{i=1}^k \frac{E^{(t)}_{1,i} }{P_{t,i}}\\
\sum_{i=1}^k \frac{E^{(t)}_{2,i} }{P_{t,i}} \\
\vdots \\
\sum_{i=1}^k \frac{E^{(t)}_{m,i} }{P_{t,i}} \\
\end{bmatrix}\\
\end{align}
$$

where the inequality is by that $E^{(t)}_{m,i} \in [0,1]$ and $z_{t,i} \in [0,1]$ for $\forall t \in [n], i \in [k], m\in [M]$

Then we have:

$$
\begin{align}
\mathbb{E}_t [Q_t\cdot (\vec{1}-\tilde{X}_{t})^{(2)}]
&= \mathbb{E}_t [Q_t \tilde{Z}_{t}^{(2)}] \\
&\le \mathbb{E}_t\left[Q_t\begin{bmatrix}
\sum_{i=1}^k \frac{E^{(t)}_{1,i} }{P_{t,i}}\\
\sum_{i=1}^k \frac{E^{(t)}_{2,i} }{P_{t,i}} \\
\vdots \\
\sum_{i=1}^k \frac{E^{(t)}_{m,i} }{P_{t,i}} \\
\end{bmatrix}\right]\\
&= \mathbb{E}_t\left[\sum_{m=1}^M Q_{t,m} \sum_{i=1}^k \frac{E^{(t)}_{m,i} }{P_{t,i}}\right]\\
&= \mathbb{E}_t\!\left[
\sum_{i=1}^{k}
\frac{
\sum_{m=1}^{M} Q_{t m} E^{(t)}_{m i}
}{
P_{t i}
}
\right] \\
&= k
\end{align}
$$

> **❓Question: How to prove the inequality holds?**
>
> does this generally hold that:
>
> $\mathbb{E}_t[Y_i] \le U_i (\forall i \in [k])\Rightarrow \mathbb{E}_t[\sum_{i=1}^k X_iY_i] \le \mathbb{E}_t[\sum_{i=1}^k X_iU_i]$
>
> Answer: If $X_i \overset{a.s.}{\ge} 0$ and $X_i$ is $\mathcal{F}_{t-1}$-measurable,then yes.
>
> $proof.$
>
> Recall that 
> - $\mathcal{F}_t := \sigma (E^{(1)}, A_1, E^{(2)}, A_2, \cdots, A_{t-1}, E^{(t)})$
> - $\mathbb{E}_t[\cdot] := \mathbb{E}_t[\cdot | \mathcal{F}_t]$
> 
> then if $X_i$ is $\mathcal{F}_{t-1}$-measurable, we have:
> $$\mathbb{E}_t[\sum_{i=1}^k X_iY_i] = \sum_{i=1}^k X_i\mathbb{E}_t[Y_i] \le \sum_{i=1}^k X_iU_i$$
> because $X_i$ are known constants at step $t$. $\blacksquare$
>
> Thus, recall that $Q_t$ has been determined at step $t-1$, thus it's $\mathcal{F}_{t-1}$-measurabl, hence the inequality 
> $$\mathbb{E}_t [Q_t \tilde{Z}_{t}^{(2)}] \le \mathbb{E}_t\left[Q_t\begin{bmatrix}\sum_{i=1}^k \frac{E^{(t)}_{1,i} }{P_{t,i}}\\\sum_{i=1}^k \frac{E^{(t)}_{2,i} }{P_{t,i}} \\\vdots \\\sum_{i=1}^k \frac{E^{(t)}_{m,i} }{P_{t,i}} \\\end{bmatrix}\right]\\$$
> holds.

Since it has been proved that $\mathbb{E}_t [Q_t\cdot (\vec{1}-\tilde{X}_{t})^{(2)}] \le k$, then we have:

$$
\begin{align}
R_n &\le \frac{\log(M)}{\eta} + \frac{\eta}{2} \sum_{t=1}^n \mathbb{E}_t [Q_t\cdot (\vec{1}-\tilde{X}_{t})^{(2)}] \\
&\le \frac{\log(M)}{\eta} + \frac{\eta n k}{2} \\
&= \sqrt{2nk\log(M)}
\end{align}
$$

The last equation comes from that we set $\eta  = \sqrt{\frac{2\log(M)}{nk}}$,and it's easy to prove that this is actualy the $\arg\min\limits_{\eta \in \mathbb{R}} \left(\frac{\log(M)}{\eta} + \frac{\eta n k}{2}\right)$

Comlete the proof. $\blacksquare$

### **3.1.6 The reason that Exp4 can't achieve $O(\log(n))$ regret upper bound** 

### **3.1.7 How the wellness of expert influence the regret upperbound**

# **4. Other Analysis**

## **4.1 The upper-bound of $R_n$**

if $|C| < \infty$, then we can decomposite $R_n$ into separate parts:

$$
R_{nc} = \mathbb{E}\left[ \max \limits_{i \in [k]} \sum_{t \in [n]: c_t = c}(x_{ti} - X_t) \right] 
$$

for $c \in C$.

Then the problem comes into multiple independent adversarial bandit problem, each of those is under a fixed context $c \in C$. Then we can apply independent Exp3 algorthims to specialize for each of $c \in C$, i.e., the first Exp3 only be responsable for $c_t = c_1$, the  second Exp3 only be responsable for $c_t = c_2$, etc. and each Exp3 doesn't share statistic information. 

Then by Exp3 relative theorems, we have:

$$
R_{nc} \le 2 \sqrt{k \left[\sum_{t=1}^n \mathbb{I}\{c_t = c\}\right] \log(k)}
$$

> **📌(1) Discusion: here we should choose an *anytime version of Exp3***
>
> Since each  $\sum_{t=1}^n \mathbb{I}\{c_t = c\}$ is unkown for the learner, so here we should choose the version of Exp3 that doesn't take the horizon as an parameter, which is called the anytime version of Exp3.

> **📌 (2) Discusion: what if $\sum_{t=1}^n \mathbb{I}\{c_t = c\}$ is small for some $c \in C$**
>
> If $\sum_{t=1}^n \mathbb{I}\{c_t = c\}$ is small for some $c \in C$, then the Exp3 w.r.t to this context $c$ might not get sufficient samples to learn, and consiquently performs badly. However, on another side of the coin, if $\sum_{t=1}^n \mathbb{I}\{c_t = c\}$ is small, then $R_{nc}$ is also small, hence cases like this wouldn't make a great contribution to the total regret $R_n = \sum_{c \in C} R_{nc}$.

Then the total regret is 

$$
R_n = \sum_{c \in C} R_{nc} \le 2 \sum_{c \in C} \sqrt{k \log(k) \sum_{t=1}^n \mathbb{I}\{c_t = c\}}
$$

rewrite the regret in 1.5 to 

$$
R_n = \mathbb{E}\left[\max\limits_{\phi \in \Phi} \sum_{t=1}^n (x_{t,\phi(c_t)} - X_t)\right]
$$

where $\Phi$ is a set of functions $\phi: C \rightarrow [k]$

this is left for the further discusion 